In [ ]:
from datasets import load_dataset

ds = load_dataset("stingning/ultrachat", split="train")

print(ds[0])
print(ds.features)

In [8]:
from datasets import load_dataset
import json

KEYWORDS = [
    "loan", "credit", "debt", "interest",
    "underwriting", "risk", "default",
    "fraud", "bank", "finance",
    "income", "repayment", "lending",
    "borrower", "credit score",
    "emi", "apr"
]

dataset = load_dataset("stingning/ultrachat", split="train")


In [9]:
# V1 (70000) samples
def contains_keyword(text):
    if not text:
        return False
    text = text.lower()
    return any(k in text for k in KEYWORDS)

filtered = []

for ex in dataset:
    # Case 1: messages format
    data = ex.get('data', [])
    if len(data) < 2:
        continue

    instruction = data[0].strip()
    response = data[1].strip()

    if contains_keyword(instruction) or contains_keyword(response):
        # length sanity check
        if 50 < len(response) < 600:
            filtered.append({
                "instruction": instruction,
                "response": response
            })

print(f"Collected {len(filtered)} candidates")

with open("sft_candidates.jsonl", "w") as f:
    for row in filtered[:500]:
        f.write(json.dumps(row) + "\n")


Collected 70604 candidates


In [ ]:
# V2 samples (750)
import re

def keyword_count(text):
    text = text.lower()
    return sum(1 for k in KEYWORDS if k in text)

def is_clean(text):
    if "```" in text:
        return False
    if len(re.findall(r"http[s]?://", text)) > 0:
        return False
    return True

filtered = []

for ex in dataset:
    # Case 1: messages format
    data = ex.get('data', [])
    if len(data) < 2:
        continue

    instruction = data[0].strip()
    response = data[1].strip()

    if not (20 <= len(instruction) <= 200):
        continue

    if not (len(response) <= 600):
       continue

    
    if keyword_count(instruction + response) < 2:
        continue

    if not is_clean(instruction + response):
        continue

    filtered.append({
        "instruction": instruction,
        "response": response
    })

print(f"Collected {len(filtered)} candidates")

with open("sft_candidates_v2.jsonl", "w") as f:
    for row in filtered:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

Collected 742 candidates


In [ ]:
# V3 Samples (Clean up)
import json

BAD_KEYWORDS = [
    "as an ai",
    "language model",
    "i cannot provide",
    "i am not able to",
    "openai",
    "chatgpt",
    "medical advice",
    "legal advice",
    "consult a doctor",
    "consult a lawyer",
    "dating",
    "astrology",
    "horoscope",
    "religion",
    "politics",
    "gaming",
    "recipe",
    "cooking",
    "subscribe",
    "click here",
    "promo",
]

def contains_bad(text):
    t = text.lower()
    return any(bad in t for bad in BAD_KEYWORDS)

clean = []
removed = []

with open("sft_candidates_v2.jsonl") as f:
    for line in f:
        row = json.loads(line)
        text = row["instruction"] + " " + row["response"]

        if contains_bad(text):
            removed.append(row)
        else:
            clean.append(row)

print(f"Kept: {len(clean)}")
print(f"Removed: {len(removed)}")

with open("sft.jsonl", "w") as f:
    for r in clean:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")


Kept: 484
Removed: 258
